# <center><span style="color:RED"><u><br/>Nettoyage et fusion des données : Transfermarkt et FBref</u></span></center>

## <span style="color:RED">1-Introduction</span>

L'objectif de ce notebook est de fusionner nos deux tables de données qui se trouvent dans le dossier data/raw.

À savoir que nous disposons de deux sources de données, qui sont les suivantes :

- fbref.csv : une table qui contient les données relatives aux performances des joueurs de Premier League durant la saison 2025-2026, disponibles sur FBref. Pour chaque joueur, nous disposons de son nom et prénom, de sa nationalité, de son poste, de son équipe, de son âge, de son année de naissance et de statistiques sur ses performances, comme le nombre de buts marqués, le nombre de minutes jouées, etc. Les données ont été téléchargées depuis cette page : https://fbref.com/en/comps/9/2025-2026/stats/2025-2026-Premier-League-Stats

- premier_league_2025_2026.csv : une table qui est le résultat de notre scraping effectué dans le notebook intitulé 01_scrapping_transfertmarkt. Pour avoir plus d'informations sur cette table, veuillez consulter le notebook en question.

Pour ce faire, voici la démarche que nous allons adopter. Nous allons importer les données, puis effectuer un premier nettoyage de chaque base avant la fusion, afin de déterminer la variable de fusion. Ensuite, nous allons gérer les éventuels doublons et les valeurs manquantes, puis fusionner nos deux bases de données avant de procéder à un nettoyage et à une vérification finale.

## <span style="color:RED">2-Importation des bibliothèques</span>

In [36]:
import pandas as pd
import numpy as np

## <span style="color:RED">3-Importation des données</span>

In [37]:
df_fbref=pd.read_csv('../../data/raw/fbref.csv')
df_premier_league=pd.read_csv('../../data/raw/premier_league_2025_2026.csv')

## <span style="color:RED">4-Nettoyage de chaque base</span>

Dans un premier temps, nous allons donc, comme annoncé, nettoyer chaque base de manière indépendante pour avoir, une fois la fusion des données, quelque chose de plus ou moins propre.

## <span style="color:RED">4.a-Nettoyage des données Fbref</span>

In [38]:
df_fbref.head(15)

,Rk,Player,Nation,Pos,Squad,Age,Born,MP,Starts,Min,...,PKatt,CrdY,CrdR,Gls.1,Ast.1,G+A.1,G-PK.1,G+A-PK,Matches,-9999
0,1,Brenden Aaronson,us USA,MFFW,Leeds United,24,2000,37,30,2449,...,0,3,0,0.15,0.18,0.33,0.15,0.33,Matches,5bc43860
1,2,Jerome Abbey,eng ENG,MF,Wolves,15,2009,1,0,17,...,0,0,0,0.00,0.00,0.00,0.00,0.00,Matches,4f504fcc
2,3,Zach Abbott,eng ENG,DF,Nottingham,19,2006,3,2,164,...,0,0,0,0.00,0.00,0.00,0.00,0.00,Matches,c103921a
3,4,Tammy Abraham,eng ENG,FW,Aston Villa,27,1997,12,2,286,...,0,0,0,0.63,0.00,0.63,0.63,0.63,Matches,f586779e
4,5,Joshua Acheampong,eng ENG,DF,Chelsea,19,2006,17,8,666,...,0,1,0,0.14,0.00,0.14,0.14,0.14,Matches,b7e62e1d
5,6,Tyler Adams,us USA,MF,Bournemouth,26,1999,25,21,1772,...,0,8,0,0.10,0.10,0.20,0.10,0.20,Matches,2b09d998
6,7,Tosin Adarabioyo,eng ENG,DF,Chelsea,27,1997,16,8,795,...,0,2,0,0.00,0.00,0.00,0.00,0.00,Matches,c81d773d
7,8,Simon Adingra,ci CIV,MFFW,Sunderland,23,2002,14,9,653,...,0,2,0,0.14,0.00,0.14,0.14,0.14,Matches,4dcec659
8,9,Amine Adli,ma MAR,MF,Bournemouth,25,2000,31,10,1094,...,0,2,0,0.25,0.08,0.33,0.25,0.33,Matches,dcfb17f4
9,10,Emmanuel Agbadou,ci CIV,DF,Wolves,28,1997,14,12,1101,...,0,1,1,0.00,0.00,0.00,0.00,0.00,Matches,75f1ed80


In [39]:
df_fbref.info()

<class 'pandas.DataFrame'>
RangeIndex: 551 entries, 0 to 550
Data columns (total 26 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Rk       551 non-null    int64  
 1   Player   551 non-null    str    
 2   Nation   551 non-null    str    
 3   Pos      551 non-null    str    
 4   Squad    551 non-null    str    
 5   Age      551 non-null    int64  
 6   Born     551 non-null    int64  
 7   MP       551 non-null    int64  
 8   Starts   551 non-null    int64  
 9   Min      551 non-null    int64  
 10  90s      551 non-null    float64
 11  Gls      551 non-null    int64  
 12  Ast      551 non-null    int64  
 13  G+A      551 non-null    int64  
 14  G-PK     551 non-null    int64  
 15  PK       551 non-null    int64  
 16  PKatt    551 non-null    int64  
 17  CrdY     551 non-null    int64  
 18  CrdR     551 non-null    int64  
 19  Gls.1    551 non-null    float64
 20  Ast.1    551 non-null    float64
 21  G+A.1    551 non-null    fl

À l'aide des deux commandes ci-dessus, nous pouvons déjà effectuer un premier constat sur la table FBref :
- La table contient 26 colonnes et 551 lignes (observations).
- Les variables numériques, comme l'âge, l'année de naissance, le nombre de matchs commencés, le nombre de buts marqués, etc., sont au bon format, à savoir int64 ou float64. Il n'est donc pas nécessaire de les convertir.
- Nous disposons de deux colonnes qui semblent inutiles : Rk, qui correspond au rang du joueur dans le tableau, et Matches, qui n'apporte aucune information supplémentaire. Il faudra donc les supprimer.
- Nous disposons également d'une colonne nommée -9999, qui contient en réalité un identifiant propre à chaque joueur. Il faudra donc renommer cette colonne pour que son nom soit plus explicite.
- Il semble n'y avoir aucune valeur manquante, mais il faudra effectuer une vérification plus précise, car certaines ont peut-être été remplacées par des zéros ou d'autres valeurs.
- Les noms des joueurs semblent être correctement affichés, même lorsqu'ils comportent des accents.
- La colonne Nation est mal formatée : elle contient deux fois la nationalité du joueur, sous deux formats différents. Par exemple, pour la première observation, nous avons us USA. Il faut donc conserver uniquement les trois dernières lettres de cette colonne.
Nous allons donc créer une fonction pour effectuer les opérations que nous avons énoncées, c'est-à-dire supprimer les colonnes Rk et Matches, puis renommer la colonne -9999 en id_fbref, puis enregistrer le DataFrame nettoyé.

In [40]:
def nettoyage_fbref(df_fbref):
    """
    Nettoie les données FBref et les enregistre au format CSV.

    Args:
        df_fbref (pd.DataFrame): Table FBref à nettoyer.

    Returns:
        pd.DataFrame: Table FBref nettoyée.
    """
    df_fbref_clean=df_fbref.drop(columns=["Rk","Matches"])
    df_fbref_clean=df_fbref_clean.rename(columns={"-9999":"id_fbref"})
    df_fbref_clean["Nation"]=df_fbref_clean["Nation"].str.strip().str[-3:]
    df_fbref_clean.to_csv("../../data/interim/fbref_clean.csv",index=False)
    return df_fbref_clean

In [41]:
df_fbref_clean=nettoyage_fbref(df_fbref)

In [42]:
df_fbref_clean.head()

,Player,Nation,Pos,Squad,Age,Born,MP,Starts,Min,90s,...,PK,PKatt,CrdY,CrdR,Gls.1,Ast.1,G+A.1,G-PK.1,G+A-PK,id_fbref
0,Brenden Aaronson,USA,MFFW,Leeds United,24,2000,37,30,2449,27.2,...,0,0,3,0,0.15,0.18,0.33,0.15,0.33,5bc43860
1,Jerome Abbey,ENG,MF,Wolves,15,2009,1,0,17,0.2,...,0,0,0,0,0.00,0.00,0.00,0.00,0.00,4f504fcc
2,Zach Abbott,ENG,DF,Nottingham,19,2006,3,2,164,1.8,...,0,0,0,0,0.00,0.00,0.00,0.00,0.00,c103921a
3,Tammy Abraham,ENG,FW,Aston Villa,27,1997,12,2,286,3.2,...,0,0,0,0,0.63,0.00,0.63,0.63,0.63,f586779e
4,Joshua Acheampong,ENG,DF,Chelsea,19,2006,17,8,666,7.4,...,0,0,1,0,0.14,0.00,0.14,0.14,0.14,b7e62e1d


In [43]:
df_fbref_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 551 entries, 0 to 550
Data columns (total 24 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Player    551 non-null    str    
 1   Nation    551 non-null    str    
 2   Pos       551 non-null    str    
 3   Squad     551 non-null    str    
 4   Age       551 non-null    int64  
 5   Born      551 non-null    int64  
 6   MP        551 non-null    int64  
 7   Starts    551 non-null    int64  
 8   Min       551 non-null    int64  
 9   90s       551 non-null    float64
 10  Gls       551 non-null    int64  
 11  Ast       551 non-null    int64  
 12  G+A       551 non-null    int64  
 13  G-PK      551 non-null    int64  
 14  PK        551 non-null    int64  
 15  PKatt     551 non-null    int64  
 16  CrdY      551 non-null    int64  
 17  CrdR      551 non-null    int64  
 18  Gls.1     551 non-null    float64
 19  Ast.1     551 non-null    float64
 20  G+A.1     551 non-null    float64
 21  G-PK

La table FBref a donc été nettoyée et enregistrée sous le nom de fbref_clean dans le fichier data/interim qui contient les tables intermédiaires, nettoyées. À présent, nous allons faire pareil pour la table premier_league_2025_2026.csv.

## <span style="color:RED">4.b-Nettoyage des données premier_league_2025_2026</span>

In [44]:
df_premier_league.head(15)

,club,joueur,age,nationalite,valeur_marchande
0,Manchester City,Gianluigi Donnarumma,27,Italy,€45.00m
1,Manchester City,James Trafford,23,England,€25.00m
2,Manchester City,Ederson,32,Brazil,€10.00m
3,Manchester City,Stefan Ortega,33,Germany,€4.00m
4,Manchester City,Marcus Bettinelli,34,England,€300k
5,Manchester City,Marc Guéhi,25,England,€70.00m
6,Manchester City,Rúben Dias,29,Portugal,€55.00m
7,Manchester City,Abdukodir Khusanov,22,Uzbekistan,€50.00m
8,Manchester City,Manuel Akanji,30,Switzerland,€17.00m
9,Manchester City,John Stones,32,England,€12.00m


In [45]:
df_premier_league.info()

<class 'pandas.DataFrame'>
RangeIndex: 798 entries, 0 to 797
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   club              798 non-null    str  
 1   joueur            798 non-null    str  
 2   age               798 non-null    int64
 3   nationalite       798 non-null    str  
 4   valeur_marchande  798 non-null    str  
dtypes: int64(1), str(4)
memory usage: 64.2 KB


À l'aide des deux commandes ci-dessus, nous pouvons également effectuer un premier constat sur la table `premier_league_2025_2026` :

- La table contient 5 colonnes et 798 lignes (observations).
- Les variables numériques, comme l'âge et la valeur marchande, ne sont pas au bon format. Il faudra donc les convertir en types numériques adaptés.
- Toutes les colonnes semblent utiles, nous n'allons donc en supprimer aucune.
- Nous remarquons que certaines valeurs marchandes sont manquantes. Cependant, étant donné qu'il s'agit de notre variable à expliquer, nous décidons de ne pas les imputer, car cela pourrait introduire un biais dans notre analyse. Nous allons donc supprimer les observations pour lesquelles cette information n'est pas disponible.
- Les noms des joueurs semblent être correctement affichés, même lorsqu'ils comportent des accents.

Nous allons donc créer une fonction pour effectuer les opérations que nous avons énoncées, c'est-à-dire convertir les colonnes `age` et `valeur_marchande` en types numériques, supprimer les observations pour lesquelles la valeur marchande est manquante, puis enregistrer le DataFrame nettoyé.

In [46]:
def nettoyage_transfermarkt(df_premier_league_2025_2026):
    """
    Nettoie les données Transfermarkt et les enregistre au format CSV.

    Args:
        df_premier_league_2025_2026 (pd.DataFrame): Table Transfermarkt à nettoyer.

    Returns:
        pd.DataFrame: Table Transfermarkt nettoyée.
    """
    df_clean=df_premier_league_2025_2026.copy()
    df_clean["age"]=pd.to_numeric(df_clean["age"],errors="coerce")
    df_clean["valeur_marchande"]=(
        df_clean["valeur_marchande"]
        .str.replace("€","",regex=False)
        .str.replace("m","e6",regex=False)
        .str.replace("k","e3",regex=False)
        .replace("-",None)
        .pipe(pd.to_numeric,errors="coerce")
        .astype("Int64")
    )
    df_clean=df_clean.dropna(subset=["valeur_marchande"])
    df_clean.to_csv("../../data/interim/transfermarkt_clean.csv",index=False)

    return df_clean

In [47]:
df_premier_league_2025_2026_clean=nettoyage_transfermarkt(df_premier_league)

In [48]:
df_premier_league_2025_2026_clean.head()

,club,joueur,age,nationalite,valeur_marchande
0,Manchester City,Gianluigi Donnarumma,27,Italy,45000000
1,Manchester City,James Trafford,23,England,25000000
2,Manchester City,Ederson,32,Brazil,10000000
3,Manchester City,Stefan Ortega,33,Germany,4000000
4,Manchester City,Marcus Bettinelli,34,England,300000


In [49]:
df_premier_league_2025_2026_clean.info()

<class 'pandas.DataFrame'>
Index: 700 entries, 0 to 796
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   club              700 non-null    str  
 1   joueur            700 non-null    str  
 2   age               700 non-null    int64
 3   nationalite       700 non-null    str  
 4   valeur_marchande  700 non-null    Int64
dtypes: Int64(1), int64(1), str(3)
memory usage: 57.6 KB


Nous venons d'effectuer un premier nettoyage des deux tables. Suite à cela, on se rend compte que la clé permettant la fusion est l'identité du joueur. En effet, un joueur est identifié par son nom et prénom et c'est l'information qui va nous permettre de fusionner les deux tables. C'est ce que nous allons faire dans la suite du Notebook.

## <span style="color:RED">5-Gestion des doublons</span>

## <span style="color:RED">5.a-Repérage des doublons FBref</span>

Dans un premier temps, nous allons identifier les joueurs présents plusieurs fois dans notre table FBref.

Ces doublons peuvent notamment s'expliquer par les transferts effectués durant la saison 2025-2026. En effet, lorsqu'un joueur a évolué dans plusieurs clubs, ses statistiques sont réparties sur plusieurs observations.

Nous allons donc repérer ces joueurs à l'aide de leur identifiant unique id_fbref.

In [50]:
doublons_fbref=df_fbref_clean[df_fbref_clean.duplicated(subset=["id_fbref"], keep=False)]
doublons_fbref.sort_values("Player")

,Player,Nation,Pos,Squad,Age,Born,MP,Starts,Min,90s,...,PK,PKatt,CrdY,CrdR,Gls.1,Ast.1,G+A.1,G-PK.1,G+A-PK,id_fbref
499,Adama Traoré,ESP,MF,Fulham,29,1996,15,1,274,3.0,...,0,0,0,0,0.00,0.00,0.00,0.00,0.00,9a28eba4
500,Adama Traoré,ESP,FWMF,West Ham,29,1996,9,0,121,1.3,...,0,0,1,0,0.00,0.00,0.00,0.00,0.00,9a28eba4
456,Antoine Semenyo,GHA,MFFW,Manchester City,25,2000,17,17,1397,15.5,...,0,0,1,0,0.45,0.06,0.52,0.45,0.52,efd2ec23
457,Antoine Semenyo,GHA,MF,Bournemouth,25,2000,20,20,1798,20.0,...,1,2,6,0,0.50,0.15,0.65,0.45,0.60,efd2ec23
254,Brennan Johnson,WAL,MFFW,Tottenham,24,2001,16,6,610,6.8,...,0,0,2,0,0.30,0.00,0.30,0.30,0.30,0cd31129
255,Brennan Johnson,WAL,MFFW,Crystal Palace,24,2001,18,13,1112,12.4,...,0,0,4,0,0.00,0.08,0.08,0.00,0.08,0cd31129
309,Douglas Luiz,BRA,MF,Aston Villa,27,1998,13,7,604,6.7,...,0,0,1,0,0.15,0.00,0.15,0.15,0.15,6f7d826d
310,Douglas Luiz,BRA,MF,Nottingham,27,1998,8,5,315,3.5,...,0,0,2,0,0.00,0.00,0.00,0.00,0.00,6f7d826d
155,Eberechi Eze,ENG,MF,Crystal Palace,27,1998,1,1,83,0.9,...,0,0,0,0,0.00,0.00,0.00,0.00,0.00,ae4fc6a4
156,Eberechi Eze,ENG,MFFW,Arsenal,27,1998,32,21,1812,20.1,...,0,0,1,0,0.35,0.10,0.45,0.35,0.45,ae4fc6a4


In [51]:
print("Nombre de joueurs en doublon :",doublons_fbref["id_fbref"].nunique())
print("Nombre de lignes concernées :",len(doublons_fbref))

Nombre de joueurs en doublon : 14
Nombre de lignes concernées : 28


## <span style="color:RED">5.b-Repérage des doublons Transfermarkt</span>

Nous allons maintenant identifier les joueurs apparaissant plusieurs fois dans notre table Transfermarkt.

Contrairement à FBref, notre table Transfermarkt ne contient pas d'identifiant unique pour chaque joueur. Nous allons donc rechercher les doublons à partir de la variable joueur.

Nous avons précédemment remarqué que certains joueurs apparaissent dans plusieurs clubs, alors que leurs autres caractéristiques, en particulier leur valeur marchande, restent identiques.

In [52]:
doublons_transfermarkt=df_premier_league_2025_2026_clean[df_premier_league_2025_2026_clean.duplicated(subset=["joueur"],keep=False)]

doublons_transfermarkt.sort_values("joueur")

,club,joueur,age,nationalite,valeur_marchande
567,West Ham United,Adama Traoré,30,Spain,6000000
718,Fulham FC,Adama Traoré,30,Spain,6000000
38,Manchester City,Antoine Semenyo,26,Ghana,80000000
359,AFC Bournemouth,Antoine Semenyo,26,Ghana,80000000
161,Liverpool FC,Ben Gannon-Doak,20,Scotland,15000000
361,AFC Bournemouth,Ben Gannon-Doak,20,Scotland,15000000
208,Tottenham Hotspur,Brennan Johnson,25,Wales,30000000
278,Crystal Palace,Brennan Johnson,25,Wales,30000000
388,Nottingham Forest,Douglas Luiz,28,Brazil,18000000
472,Aston Villa,Douglas Luiz,28,Brazil,18000000


In [53]:
print("Nombre de noms en doublon :",doublons_transfermarkt["joueur"].nunique())
print("Nombre de lignes concernées :",len(doublons_transfermarkt))

Nombre de noms en doublon : 17
Nombre de lignes concernées : 34


## <span style="color:RED">5.c-Regroupement des doublons FBref</span>

La partie 5.a nous montre que certains joueurs apparaissent plusieurs fois dans notre table. Cependant, ces doublons ne sont en aucun cas des erreurs. Nous avons en effet 14 joueurs qui apparaissent deux fois. Cela s'explique par des transferts d'un club de Premier League vers un autre club de Premier League durant la saison 2025-2026. Par exemple, le premier doublon est Adama Traoré : il a commencé la saison à Fulham, puis a été transféré à West Ham. Il faut donc traiter ces cas particuliers afin d'obtenir une seule observation par joueur.

Pour gérer ces doublons, nous allons regrouper leurs statistiques afin d'obtenir une seule observation par joueur. Pour cela, nous allons additionner les statistiques cumulables, comme les minutes jouées, les buts et les passes décisives. Par exemple, si un joueur A a marqué 2 buts dans le club 1, puis 7 buts dans le club 2, il aura marqué 9 buts au total.

Cependant, les statistiques exprimées par 90 minutes ne peuvent pas être additionnées directement. Nous allons donc les recalculer à partir des statistiques totales.

Enfin, afin de conserver une seule observation par joueur, nous retiendrons le club dans lequel le joueur a disputé le plus de minutes durant la saison. Bien évidemment, d'autres choix sont possibles, comme conserver le dernier club dans lequel le joueur a évolué. Ce choix méthodologique reste donc discutable.

In [54]:
def regroupement_fbref(df):
    """
    Regroupe les statistiques des joueurs présents plusieurs fois.

    Args:
        df (pd.DataFrame): Table FBref nettoyée.

    Returns:
        pd.DataFrame: Table FBref avec une seule ligne par joueur.
    """
    colonnes_a_sommer=["MP","Starts","Min","Gls","Ast","G+A","G-PK","PK","PKatt","CrdY","CrdR"]
    df=df.sort_values("Min",ascending=False)
    df_regroupe=df.groupby("id_fbref",as_index=False).agg({
        "Player":"first",
        "Nation":"first",
        "Pos":"first",
        "Squad":"first",
        "Age":"first",
        "Born":"first",
        **{col:"sum" for col in colonnes_a_sommer}
    })
    df_regroupe["90s"]=df_regroupe["Min"]/90
    ratios={"Gls.1":"Gls","Ast.1":"Ast","G+A.1":"G+A","G-PK.1":"G-PK"}
    for ratio, total in ratios.items():
        df_regroupe[ratio]=(df_regroupe[total]/df_regroupe["90s"])

    df_regroupe["G+A-PK"]=((df_regroupe["G-PK"]+df_regroupe["Ast"])/df_regroupe["90s"])

    return df_regroupe

In [55]:
df_fbref_regroupe=regroupement_fbref(df_fbref_clean)

df_fbref_regroupe.head()

,id_fbref,Player,Nation,Pos,Squad,Age,Born,MP,Starts,Min,...,PK,PKatt,CrdY,CrdR,90s,Gls.1,Ast.1,G+A.1,G-PK.1,G+A-PK
0,0041f3e6,Mohamadou Kanté,FRA,MF,West Ham,19,2005,11,0,75,...,0,0,2,0,0.833333,0.000000,0.0,0.000000,0.000000,0.000000
1,00963611,Konstantinos Mavropanos,GRE,DF,West Ham,27,1997,31,27,2534,...,0,0,2,0,28.155556,0.106551,0.0,0.106551,0.106551,0.106551
2,0107757e,Filip Jørgensen,DEN,GK,Chelsea,23,2002,5,3,379,...,0,0,0,0,4.211111,0.000000,0.0,0.000000,0.000000,0.000000
3,013d348e,Melker Ellborg,SWE,GK,Sunderland,22,2003,3,3,270,...,0,0,0,0,3.000000,0.000000,0.0,0.000000,0.000000,0.000000
4,01834aa0,Sebastiaan Bornauw,BEL,DF,Leeds United,26,1999,12,5,429,...,0,0,1,0,4.766667,0.000000,0.0,0.000000,0.000000,0.000000


In [56]:
print("Dimensions :",df_fbref_regroupe.shape)

print("Doublons restants :",df_fbref_regroupe.duplicated(subset=["id_fbref"]).sum())

Dimensions : (537, 24)
Doublons restants : 0


In [57]:
print(df_fbref_regroupe[df_fbref_regroupe["Player"]=="Antoine Semenyo"][["Player","MP","Min","Gls","Ast","Gls.1"]])

              Player  MP   Min  Gls  Ast     Gls.1
512  Antoine Semenyo  37  3195   17    4  0.478873


Ainsi, nous venons de gérer les doublons dans la table FBref. À présent, il faut gérer ceux de la table Transfermarkt.

## <span style="color:RED">5.4. Traitement des doublons Transfermarkt</span>


Nous avons constaté que 17 noms de joueurs apparaissent deux fois dans notre table Transfermarkt, notamment en raison des transferts effectués entre plusieurs clubs de Premier League durant la saison 2025-2026.

Contrairement à FBref, les observations en doublon semblent contenir les mêmes informations concernant l'âge, la nationalité et la valeur marchande. Il n'y a que pour le club que cela diffère.

Nous allons donc vérifier que ces informations sont bien identiques avant de supprimer les observations supplémentaires.

Concernant le club, nous avons précédemment choisi de conserver, dans FBref, celui dans lequel le joueur a disputé le plus de minutes. Nous utiliserons donc cette information lors de la fusion finale afin de conserver une règle cohérente pour l'ensemble de notre base de données.

In [58]:
def traitement_doublons_transfermarkt(df):
    """
    Supprime les doublons de la table Transfermarkt.

    Args:
        df (pd.DataFrame): Table Transfermarkt nettoyée.

    Returns:
        pd.DataFrame: Table Transfermarkt sans doublons.
    """
    df_clean=df.copy()
    df_clean=df_clean.drop(columns=["club"])
    df_clean=df_clean.drop_duplicates(subset=["joueur"])
    return df_clean

In [59]:
df_transfermarkt_unique=traitement_doublons_transfermarkt(df_premier_league_2025_2026_clean)

print("Dimensions :",df_transfermarkt_unique.shape)
print("Doublons restants :",df_transfermarkt_unique.duplicated(subset=["joueur"]).sum())

Dimensions : (683, 4)
Doublons restants : 0


## <span style="color:RED">5-Fusion des deux bases de données</span>

Dans cette partie, nous allons fusionner les deux tables à l'aide de l'identité des joueurs (joueur dans la table premier_league_2025_2026_clean et Player dans celle de FBref).

Nous allons uniquement conserver les observations présentes dans les deux tables à la fois, afin de limiter les valeurs manquantes et d'obtenir une base de données exploitable pour notre analyse statistique et économétrique.

In [60]:
import unicodedata

def normaliser_nom(nom):
    nom=nom.lower().strip()
    nom=unicodedata.normalize("NFKD",nom)
    nom="".join(c for c in nom if not unicodedata.combining(c))
    return nom

df_fbref_regroupe["nom_fusion"]=df_fbref_regroupe["Player"].apply(normaliser_nom)

df_transfermarkt_unique["nom_fusion"]=df_transfermarkt_unique["joueur"].apply(normaliser_nom)

In [61]:
def fusion_tables(df_fbref,df_transfermarkt):
    """
    Fusionne les données FBref et Transfermarkt
    à partir des noms normalisés.
    """
    df_merge=pd.merge(df_fbref,df_transfermarkt,on="nom_fusion",how="inner",validate="one_to_one")
    return df_merge

In [62]:
df_merge=fusion_tables(df_fbref_regroupe,df_transfermarkt_unique)

print("Dimensions :",df_merge.shape)
print("Joueurs distincts :",df_merge["id_fbref"].nunique())
print("Doublons :",df_merge.duplicated(subset=["id_fbref"]).sum())

Dimensions : (505, 29)
Joueurs distincts : 505
Doublons : 0


In [63]:
df_merge.loc[df_merge["Age"]!=df_merge["age"],["Player", "Age", "age"]].head(15)

,Player,Age,age
0,Mohamadou Kanté,19,20
1,Konstantinos Mavropanos,27,28
2,Filip Jørgensen,23,24
3,Melker Ellborg,22,23
4,Sebastiaan Bornauw,26,27
5,Rio Ngumoha,16,17
6,Joshua Zirkzee,24,25
7,Jun'ai Byfield,16,17
8,Ezri Konsa,27,28
9,Séamus Coleman,36,37


Nous obtenons ainsi une table contenant 505 lignes et 29 colonnes. Avant d'enregistrer cette table, nous allons la nettoyer, car certaines colonnes apportent les mêmes informations.
- Player et joueur contiennent la même information. Nous allons conserver Player.
- Nation et nationalite contiennent la même information. Nous allons conserver nationalite, qui est plus explicite.
- club et Squad contiennent la même information. Nous allons conserver club, qui est plus explicite.
- Age et age ne contiennent pas exactement la même information. Nous allons conserver Age, qui est celle de FBref et qui nous donne l'âge des joueurs de la saison 2025-2026, contrairement à age, qui nous donne l'âge actuel.
Nous allons donc créer une fonction permettant de supprimer les colonnes que nous ne souhaitons pas conserver et de renommer les colonnes restantes afin d'avoir des noms explicites.

In [64]:
def nettoyage_fusion(df):
    """
    Supprime les colonnes redondantes de la table fusionnée.

    Args:
        df (pd.DataFrame): Table fusionnée.

    Returns:
        pd.DataFrame: Table fusionnée nettoyée.
    """
    df_clean=df.drop(columns=["joueur","Nation","Squad","age"])
    df_clean.to_csv("../../data/interim/df_merge_clean.csv",index=False)
    return df_clean

In [65]:
df_merge_clean=nettoyage_fusion(df_merge)
df_merge_clean

,id_fbref,Player,Pos,Age,Born,MP,Starts,Min,Gls,Ast,...,CrdR,90s,Gls.1,Ast.1,G+A.1,G-PK.1,G+A-PK,nom_fusion,nationalite,valeur_marchande
0,0041f3e6,Mohamadou Kanté,MF,19,2005,11,0,75,0,0,...,0,0.833333,0.000000,0.000000,0.000000,0.000000,0.000000,mohamadou kante,France,1000000
1,00963611,Konstantinos Mavropanos,DF,27,1997,31,27,2534,3,0,...,0,28.155556,0.106551,0.000000,0.106551,0.106551,0.106551,konstantinos mavropanos,Greece,18000000
2,0107757e,Filip Jørgensen,GK,23,2002,5,3,379,0,0,...,0,4.211111,0.000000,0.000000,0.000000,0.000000,0.000000,filip jørgensen,Denmark,15000000
3,013d348e,Melker Ellborg,GK,22,2003,3,3,270,0,0,...,0,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,melker ellborg,Sweden,3000000
4,01834aa0,Sebastiaan Bornauw,DF,26,1999,12,5,429,0,0,...,0,4.766667,0.000000,0.000000,0.000000,0.000000,0.000000,sebastiaan bornauw,Belgium,8000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
500,feb5d972,Pape Matar Sarr,MF,22,2002,26,13,1398,2,4,...,0,15.533333,0.128755,0.257511,0.386266,0.128755,0.386266,pape matar sarr,Senegal,30000000
501,febb2831,Nehemiah Oriola,MF,18,2007,1,0,1,0,0,...,0,0.011111,0.000000,0.000000,0.000000,0.000000,0.000000,nehemiah oriola,England,200000
502,fed7cb61,Kai Havertz,FWMF,26,1999,12,7,581,2,3,...,0,6.455556,0.309811,0.464716,0.774527,0.309811,0.774527,kai havertz,Germany,55000000
503,ff5ea0bf,Malick Thiaw,DF,23,2001,35,33,2965,4,0,...,0,32.944444,0.121417,0.000000,0.121417,0.121417,0.121417,malick thiaw,Germany,45000000


Avant d'enregistrer cette table finale, nous allons renommer les colonnes afin d'attribuer un nom explicite à chaque variable et de faciliter leur compréhension par la suite.

In [66]:
def nettoyage_final(df):
    """
    Nettoie et enregistre la table fusionnée.

    Args:
        df (pd.DataFrame): Table fusionnée.

    Returns:
        pd.DataFrame: Table finale nettoyée.
    """
    df_clean=df.drop(columns=["nom_fusion"])
    df_clean=df_clean.rename(columns={
        "Player":"joueur",
        "Pos":"poste",
        "Age":"age",
        "Born":"annee_naissance",
        "MP":"matchs_joues",
        "Starts":"titularisations",
        "Min":"minutes_jouees",
        "Gls":"buts",
        "Ast":"passes_decisives",
        "G+A":"buts_plus_passes",
        "G-PK":"buts_hors_penalty",
        "PK":"penaltys_marques",
        "PKatt":"penaltys_tentes",
        "CrdY":"cartons_jaunes",
        "CrdR":"cartons_rouges",
        "90s":"equivalent_90_minutes",
        "Gls.1":"buts_par_90",
        "Ast.1":"passes_decisives_par_90",
        "G+A.1":"buts_plus_passes_par_90",
        "G-PK.1":"buts_hors_penalty_par_90",
        "G+A-PK":"buts_plus_passes_hors_penalty_par_90"
    })
    #Création du logarithme de la valeur marchande pour la future régréssion
    df_clean["log_valeur_marchande"]=np.log(
    df_clean["valeur_marchande"].astype(float))
    df_clean.to_csv("../../data/processed/dataset_final.csv",index=False)
    return df_clean

In [67]:
df_final=nettoyage_final(df_merge_clean)
df_final.head()

,id_fbref,joueur,poste,age,annee_naissance,matchs_joues,titularisations,minutes_jouees,buts,passes_decisives,...,cartons_rouges,equivalent_90_minutes,buts_par_90,passes_decisives_par_90,buts_plus_passes_par_90,buts_hors_penalty_par_90,buts_plus_passes_hors_penalty_par_90,nationalite,valeur_marchande,log_valeur_marchande
0,0041f3e6,Mohamadou Kanté,MF,19,2005,11,0,75,0,0,...,0,0.833333,0.000000,0.0,0.000000,0.000000,0.000000,France,1000000,13.815511
1,00963611,Konstantinos Mavropanos,DF,27,1997,31,27,2534,3,0,...,0,28.155556,0.106551,0.0,0.106551,0.106551,0.106551,Greece,18000000,16.705882
2,0107757e,Filip Jørgensen,GK,23,2002,5,3,379,0,0,...,0,4.211111,0.000000,0.0,0.000000,0.000000,0.000000,Denmark,15000000,16.523561
3,013d348e,Melker Ellborg,GK,22,2003,3,3,270,0,0,...,0,3.000000,0.000000,0.0,0.000000,0.000000,0.000000,Sweden,3000000,14.914123
4,01834aa0,Sebastiaan Bornauw,DF,26,1999,12,5,429,0,0,...,0,4.766667,0.000000,0.0,0.000000,0.000000,0.000000,Belgium,8000000,15.894952


In [68]:
valeurs_manquantes=["-","--",""," ","NA","N/A","NaN","nan","None","null","NULL"]
verification=(df_final.isna()|df_final.astype("string").apply(lambda col: col.str.strip().isin(valeurs_manquantes)))

print("Valeurs manquantes par colonne :")
print(verification.sum())

print("\nTotal :",verification.sum().sum())

Valeurs manquantes par colonne :
id_fbref                                0
joueur                                  0
poste                                   0
age                                     0
annee_naissance                         0
matchs_joues                            0
titularisations                         0
minutes_jouees                          0
buts                                    0
passes_decisives                        0
buts_plus_passes                        0
buts_hors_penalty                       0
penaltys_marques                        0
penaltys_tentes                         0
cartons_jaunes                          0
cartons_rouges                          0
equivalent_90_minutes                   0
buts_par_90                             0
passes_decisives_par_90                 0
buts_plus_passes_par_90                 0
buts_hors_penalty_par_90                0
buts_plus_passes_hors_penalty_par_90    0
nationalite                             0
v

## <span style="color:RED">Conclusion</span>

Ce notebook aboutit donc à la création de notre base de données finale, nettoyée et ne contenant aucune valeur manquante.

C'est cette base de données que nous utiliserons pour répondre à notre problématique : **« Comment expliquer la valeur marchande des joueurs de Premier League ? »**

Nous présenterons cette base de données et l'étudierons en détail dans le prochain notebook qui inclura une présentation détaillée avec des statistiques descriptives.